# 03 — Addestrare, valutare e leggere un Decision Tree

Percorso: **X/y → train_test_split → modello → fit → predict → metriche → grafici**.
Usiamo direttamente pandas e scikit-learn; da `src/` importiamo soltanto lo schema
delle colonne e la mappa documentata delle famiglie.

Conserviamo tutte le **41 feature**. Dopo baseline e riferimento a profondità 10,
confrontiamo soltanto **10, 13 e 15**, sullo stesso split. Le confusion matrix
normalizzate per riga e i conteggi delle classi rare guidano la lettura dei risultati.
Il grafico dell'albero mostra pochi livelli solo per leggibilità: le metriche usano
sempre le predizioni dell'albero completo. Nessun fit a profondità 3 o 5.

Tutte le misure provengono dalla validation interna a KDDTrain+; non leggiamo KDDTest+.


In [ ]:
from pathlib import Path
import sys
import json
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS

## 1. Leggere i dati, definire X/y e usare train_test_split

`pd.read_csv` è la funzione pandas: il CSV non ha header, quindi passiamo i nomi
delle colonne. `X` contiene le 41 feature; `y` deriva dalla mappa documentata delle
etichette tramite `Series.map`, senza inventare un ordine numerico delle famiglie.

`train_test_split` è la funzione originale sklearn: 80% train, 20% validation,
seed 42, stratificazione per famiglia. Il parametro si chiama `test_size`, ma
questa è una validation ricavata dal training; il test ufficiale resta separato.


In [ ]:
from sklearn.model_selection import train_test_split
from src.targets import ATTACK_TO_FAMILY, FAMILY_ORDER

train_path = project_root / 'data/raw/KDDTrain+.csv'
string_types = {column: 'string' for column in (*CATEGORICAL_COLUMNS, 'attack_label')}
train = pd.read_csv(train_path, header=None, names=COLUMN_NAMES, dtype=string_types)
X = train.loc[:, list(FEATURE_COLUMNS)].copy()
unknown = train['attack_label'].isna() | ~train['attack_label'].isin(ATTACK_TO_FAMILY)
if unknown.any():
    raise ValueError('Etichette mancanti o non mappate: riesaminare il target.')
y = train['attack_label'].map(ATTACK_TO_FAMILY).rename('family')

X_train, X_validation, y_train, y_validation = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print(f'Train: {len(X_train):,}; validation: {len(X_validation):,}; feature: {X.shape[1]}')


### Verificare che sia lo stesso esperimento

Il blocco seguente controlla la provenienza e confronta gli indici con quelli salvati
dal notebook 02. Non addestra il modello. Eseguire prima il notebook 02 se mancano i file.


In [ ]:
processed = project_root / 'data/processed'
metadata = json.loads((processed / 'validation_split.json').read_text(encoding='utf-8'))
assert hashlib.sha256(train_path.read_bytes()).hexdigest() == metadata['source_sha256']
assert (metadata['test_size'], metadata['random_state']) == (0.2, 42)
assert X.index.equals(pd.RangeIndex(len(X)))
train_idx = X_train.index.to_numpy()
validation_idx = X_validation.index.to_numpy()
with np.load(processed / 'validation_split.npz', allow_pickle=False) as saved:
    np.testing.assert_array_equal(train_idx, saved['train'])
    np.testing.assert_array_equal(validation_idx, saved['validation'])
assert not np.intersect1d(train_idx, validation_idx).size
print('Stesse righe e stesso ordine del notebook 02.')


## 2. Baseline: prevedere sempre la famiglia più frequente

`DummyClassifier(strategy="most_frequent")` ignora i valori delle feature e sceglie la classe
più frequente nel solo train. Non richiede one-hot. Registriamo separatamente tempo di fit e
predizione: i tempi dipendono dalla macchina e non sono un benchmark di prestazioni computazionali.

In [ ]:
from time import perf_counter
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

results, predictions = {}, {}
baseline = DummyClassifier(strategy='most_frequent')
start = perf_counter()
baseline.fit(X_train, y_train)
baseline_fit_seconds = perf_counter() - start
start = perf_counter()
predictions['baseline'] = baseline.predict(X_validation)
baseline_predict_seconds = perf_counter() - start

print('Classe sempre prevista:', np.unique(predictions['baseline']).tolist())
print(classification_report(y_validation, predictions['baseline'], labels=FAMILY_ORDER, zero_division=0, digits=4))


La baseline rende visibile il problema dello sbilanciamento: una buona quota di record normali
non implica riconoscere gli attacchi. Precision/recall/F1 delle classi mai predette sono zero
(`zero_division=0`), non valori mancanti. Riportiamo sempre anche macro F1, che pesa le cinque
famiglie in modo uguale; weighted F1 pesa invece per il loro numero di record.

## 3. Creare il modello: preprocessing + Decision Tree

Scriviamo direttamente `ColumnTransformer`, `OneHotEncoder`, `Pipeline` e
[DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html).
Il classificatore richiede input numerici: one-hot per le tre colonne testuali,
38 numeriche invariate. Nessun fit dell'encoder prima dello split.

`max_depth=10` limita il numero di archi dalla radice alla foglia più lontana
(radice = profondità 0); `random_state=42` rende ripetibile l'addestramento.
Quando chiamiamo `Pipeline.fit` apprende encoder e albero dal solo train;
`Pipeline.predict` applica l'encoder già appreso e poi il classificatore.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

numeric_columns = [column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS]
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), list(CATEGORICAL_COLUMNS)),
    ('numeric', 'passthrough', numeric_columns),
], remainder='drop')

# Riferimento originale: tutte le 41 feature e profondità 10.
tree = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=10, random_state=42)),
])


### 3.1 Addestrare: fit(X_train, y_train)

L'albero apprende le regole dalle feature e dalle risposte del train.
La validation non viene passata a `fit`.


In [ ]:
start = perf_counter()
tree.fit(X_train, y_train)
tree_fit_seconds = perf_counter() - start

classifier = tree.named_steps['classifier']
print(f'Profondità appresa: {classifier.get_depth()}; foglie: {classifier.get_n_leaves()}')


### 3.2 Prevedere: predict(X_validation)

Ora il modello riceve solo le feature di validation e produce una famiglia per riga.
Useremo `y_validation` dopo, per confrontare previsioni e risposte vere.


In [ ]:
start = perf_counter()
y_pred = tree.predict(X_validation)
tree_predict_seconds = perf_counter() - start
predictions['decision_tree'] = y_pred

display(pd.DataFrame({
    'famiglia vera': y_validation.to_numpy(), 'famiglia prevista': y_pred,
}, index=X_validation.index).head(10))


## 4. Metriche e confronto omogeneo

Stesse osservazioni di validation per entrambi. Matrici: righe = famiglia vera,
colonne = famiglia predetta, ordine **Normal, DoS, Probe, R2L, U2R**.
Support mostra quanti esempi sostengono ciascuna misura; le stime sulle classi rare sono fragili.

In [ ]:
# Funzioni originali sklearn: stessa valutazione per entrambi i modelli.
timings = {
    'baseline': (baseline_fit_seconds, baseline_predict_seconds),
    'decision_tree': (tree_fit_seconds, tree_predict_seconds),
}
for name, y_predicted in predictions.items():
    results[name] = {
        'accuracy': float(accuracy_score(y_validation, y_predicted)),
        'f1_weighted': float(f1_score(y_validation, y_predicted, labels=FAMILY_ORDER, average='weighted', zero_division=0)),
        'f1_macro': float(f1_score(y_validation, y_predicted, labels=FAMILY_ORDER, average='macro', zero_division=0)),
        'classification_report': classification_report(y_validation, y_predicted, labels=FAMILY_ORDER, output_dict=True, zero_division=0),
        'confusion_matrix': confusion_matrix(y_validation, y_predicted, labels=FAMILY_ORDER).tolist(),
        'fit_seconds': timings[name][0],
        'predict_seconds': timings[name][1],
    }

summary_columns = ['accuracy', 'f1_weighted', 'f1_macro', 'fit_seconds', 'predict_seconds']
summary = pd.DataFrame(results).T.loc[:, summary_columns].astype(float)
display(summary.round(6))
display(pd.DataFrame(results['decision_tree']['classification_report']).T.drop(index='accuracy').round(4))


In [ ]:
for name, result in results.items():
    print(name)
    display(pd.DataFrame(result['confusion_matrix'], index=FAMILY_ORDER, columns=FAMILY_ORDER))
    assert np.asarray(result['confusion_matrix']).sum() == len(y_validation)
    assert np.isfinite([result[key] for key in ['accuracy', 'f1_macro', 'f1_weighted']]).all()

### Confusion matrix: quali famiglie confondiamo?

Righe = famiglia vera, colonne = previsione. La diagonale contiene i casi corretti.
A sinistra i conteggi; a destra la normalizzazione **per riga**, per rendere visibili
anche le classi rare. Per esempio, 3 errori su 10 U2R sono il 30% della famiglia.

La confusion matrix confronta **target e predizioni**: non misura correlazioni
fra le feature e non dice quale colonna eliminare. Per questo il conteggio dei
supporti accompagna sempre il grafico delle percentuali.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for ax, normalization, title, value_format in zip(
    axes, [None, 'true'], ['Conteggi', 'Quote per famiglia vera'], ['d', '.2f'],
):
    ConfusionMatrixDisplay.from_predictions(
        y_validation, predictions['decision_tree'],
        labels=FAMILY_ORDER, display_labels=FAMILY_ORDER,
        normalize=normalization, values_format=value_format,
        cmap='Blues', colorbar=False, ax=ax,
    )
    ax.set(title=title, xlabel='Famiglia prevista', ylabel='Famiglia vera')
plt.show()


In [ ]:
tree_report = results['decision_tree']['classification_report']
for family in FAMILY_ORDER:
    row = tree_report[family]
    print(f"{family}: recall={row['recall']:.3f}, F1={row['f1-score']:.3f}, support={int(row['support'])}")
matrix = np.array(results['decision_tree']['confusion_matrix'])
np.fill_diagonal(matrix, 0)
errors = [(FAMILY_ORDER[i], FAMILY_ORDER[j], int(matrix[i, j])) for i, j in zip(*np.nonzero(matrix))]
display(pd.DataFrame(sorted(errors, key=lambda row: row[2], reverse=True), columns=['vera', 'predetta', 'errori']).head(8))

## 5. Salvare le evidenze e ripetere il percorso

Registriamo predizioni, indici, metriche, parametri e versioni in `artifacts/`, esclusa da Git.
Il riepilogo leggibile delle misure effettive è in [FIRST_MODEL_RESULTS.md](../docs/FIRST_MODEL_RESULTS.md).
Una seconda esecuzione pulita deve conservare split, predizioni e metriche; i tempi possono variare.

In [ ]:
import platform
import importlib.metadata

artifacts = project_root / 'artifacts'
artifacts.mkdir(exist_ok=True)
experiment = {
    'evaluation_set': 'validation from KDDTrain+ only', 'split': metadata,
    'versions': {'python': platform.python_version(), **{name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'scikit-learn']}},
    'parameters': {'baseline': baseline.get_params(), 'decision_tree': tree.named_steps['classifier'].get_params(), 'encoding': 'one-hot: 3 categorical; 38 numerical unchanged'},
    'results': results,
}
(artifacts / 'first_model_results.json').write_text(json.dumps(experiment, indent=2), encoding='utf-8')

In [ ]:
np.savez(artifacts / 'first_model_predictions.npz', validation_indices=validation_idx, baseline=np.asarray(predictions['baseline'], dtype=str), decision_tree=np.asarray(predictions['decision_tree'], dtype=str))
print('Evidenze locali salvate; nessuna valutazione del test ufficiale.')

## 6. Confrontare profondità 10, 13 e 15

Varia soltanto `max_depth`: stessi input, split, preprocessing e seed 42.
L'albero a profondità 10 è il riferimento già addestrato; per 13 e 15 usiamo
`clone`, la funzione sklearn che copia la configurazione senza copiare il fit.
Ogni nuovo albero e il suo encoder apprendono soltanto dal train.

Un'accuracy elevata può convivere con un recall basso sulle famiglie rare.
Perciò conserviamo report per famiglia, matrici con conteggi e matrici normalizzate
**per riga**: la diagonale corrisponde al recall, il resto della riga agli errori
su quella famiglia. Il confronto resta esplorativo sulla stessa validation;
non scegliamo automaticamente il modello finale.


In [ ]:
from sklearn.base import clone

depths_to_try = [10, 13, 15]
depth_models = {10: tree}
depth_predictions = {}
depth_reports = {}
depth_rows = []
class_rows = []

for depth in depths_to_try:
    if depth != 10:
        depth_models[depth] = clone(tree).set_params(classifier__max_depth=depth)
        depth_models[depth].fit(X_train, y_train)
    candidate = depth_models[depth]
    train_pred = candidate.predict(X_train)
    validation_pred = candidate.predict(X_validation)
    depth_predictions[depth] = validation_pred
    report = classification_report(
        y_validation, validation_pred, labels=FAMILY_ORDER, output_dict=True, zero_division=0,
    )
    counts = confusion_matrix(y_validation, validation_pred, labels=FAMILY_ORDER)
    normalized = confusion_matrix(y_validation, validation_pred, labels=FAMILY_ORDER, normalize='true')
    fitted_classifier = candidate.named_steps['classifier']
    depth_rows.append({
        'max_depth': depth,
        'profondita_appresa': fitted_classifier.get_depth(),
        'foglie': fitted_classifier.get_n_leaves(),
        'accuracy_train': accuracy_score(y_train, train_pred),
        'accuracy_validation': accuracy_score(y_validation, validation_pred),
        'macro_f1_train': f1_score(y_train, train_pred, labels=FAMILY_ORDER, average='macro', zero_division=0),
        'macro_f1_validation': f1_score(y_validation, validation_pred, labels=FAMILY_ORDER, average='macro', zero_division=0),
    })
    depth_reports[depth] = {
        'classification_report': report,
        'confusion_matrix': counts.tolist(),
        'confusion_matrix_normalized': normalized.tolist(),
        'parameters': fitted_classifier.get_params(),
    }
    for i, family in enumerate(FAMILY_ORDER):
        support = int(counts[i].sum())
        correct = int(counts[i, i])
        class_rows.append({
            'max_depth': depth, 'famiglia': family,
            'support': support, 'corretti': correct, 'errori': support - correct,
            'recall': report[family]['recall'],
            'quota_errori': (support - correct) / support,
            'precision': report[family]['precision'], 'f1': report[family]['f1-score'],
        })

depth_results = pd.DataFrame(depth_rows).set_index('max_depth')
class_results = pd.DataFrame(class_rows).set_index(['max_depth', 'famiglia'])
np.testing.assert_array_equal(depth_predictions[10], predictions['decision_tree'])
display(depth_results.round(6))
display(class_results.round(4))
depth_results.to_csv(artifacts / 'tree_depth_comparison.csv')
class_results.to_csv(artifacts / 'tree_depth_class_metrics.csv')


### Confusion matrix normalizzate: confronto delle tre profondità

Stesso ordine delle famiglie, stessa scala cromatica da 0% a 100% in tutti i pannelli.
Ogni riga somma a 100% prima dell'arrotondamento. Un valore fuori diagonale indica
quale altra famiglia riceve quegli esempi. I conteggi nel riepilogo sotto evitano
di confondere una percentuale misurata su 10 esempi con una misurata su migliaia.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for ax, depth in zip(axes, depths_to_try):
    ConfusionMatrixDisplay.from_predictions(
        y_validation, depth_predictions[depth],
        labels=FAMILY_ORDER, display_labels=FAMILY_ORDER,
        normalize='true', values_format='.1%', cmap='Blues', colorbar=False,
        im_kw={'vmin': 0, 'vmax': 1}, ax=ax,
    )
    ax.set(title=f'max_depth = {depth}', xlabel='Famiglia prevista', ylabel='Famiglia vera')
fig.suptitle('Validation: quote per famiglia vera, alberi completi')
fig.savefig(artifacts / 'tree_depth_confusion_matrices.png', dpi=160)
plt.show()


In [ ]:
rare_classes = class_results.loc[(slice(None), ['R2L', 'U2R']), :]
display(rare_classes[['support', 'corretti', 'errori', 'recall', 'quota_errori', 'precision', 'f1']].round(4))

# Conteggi completi, da leggere insieme alle percentuali dei grafici.
for depth in depths_to_try:
    print(f'Confusion matrix con conteggi: max_depth={depth}')
    display(pd.DataFrame(
        depth_reports[depth]['confusion_matrix'], index=FAMILY_ORDER, columns=FAMILY_ORDER,
    ))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, metric, title in zip(axes, ['accuracy', 'macro_f1'], ['Accuracy', 'Macro F1']):
    train_values = depth_results[f'{metric}_train']
    validation_values = depth_results[f'{metric}_validation']
    ax.plot(depth_results.index, train_values, 'o-', label='Train')
    ax.plot(depth_results.index, validation_values, 'o-', label='Validation')
    lower = max(0, min(train_values.min(), validation_values.min()) - 0.005)
    upper = min(1.001, max(train_values.max(), validation_values.max()) + 0.005)
    ax.set(xlabel='Profondità massima (max_depth)', ylabel=title, ylim=(lower, upper))
    ax.set_xticks(depths_to_try)
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle('Stesso split, profondità 10 / 13 / 15 — assi verticali ingranditi')
fig.savefig(artifacts / 'tree_depth_comparison.png', dpi=160)
plt.show()


### Salvare le misure del confronto

Le nuove misure restano separate dal riferimento iniziale: JSON con report e matrici,
CSV con metriche aggregate e per famiglia, NPZ con indici e predizioni Unicode.
Le predizioni si possono rileggere con `allow_pickle=False`.


In [ ]:
depth_experiment = {
    'evaluation_set': 'validation from KDDTrain+ only',
    'split': metadata, 'feature_count': X.shape[1],
    'family_order': list(FAMILY_ORDER), 'depths': depths_to_try,
    'versions': experiment['versions'],
    'results': {
        str(depth): {**depth_results.loc[depth].to_dict(), **depth_reports[depth]}
        for depth in depths_to_try
    },
}
(artifacts / 'tree_depth_results.json').write_text(json.dumps(depth_experiment, indent=2), encoding='utf-8')
np.savez(
    artifacts / 'tree_depth_predictions.npz',
    validation_indices=validation_idx,
    **{f'depth_{depth}': np.asarray(values, dtype=str) for depth, values in depth_predictions.items()},
)


## 7. Disegnare pochi livelli di un albero completo

`depth_to_plot` sceglie uno dei tre modelli **già addestrati** (10, 13 o 15).
`plot_max_depth=3` limita esclusivamente il disegno: non crea un albero poco profondo
e non modifica le predizioni usate per le metriche. I simboli `(...)` indicano
sottoalberi nascosti, che continuano a essere usati nella classificazione.
Per cambiare la vista basta modificare i parametri e rieseguire la cella seguente.


In [ ]:
from sklearn.tree import plot_tree

depth_to_plot = 15  # Uno dei modelli già valutati: 10, 13 o 15.
plot_max_depth = 3  # Limite del solo disegno.
tree_to_plot = depth_models[depth_to_plot]
classifier_to_plot = tree_to_plot.named_steps['classifier']
encoded_feature_names = tree_to_plot.named_steps['preprocessor'].get_feature_names_out()

fig, ax = plt.subplots(figsize=(24, 12), constrained_layout=True)
plot_tree(
    classifier_to_plot, max_depth=plot_max_depth,
    feature_names=encoded_feature_names,
    class_names=classifier_to_plot.classes_.tolist(),
    filled=True, rounded=True, proportion=True, precision=3, fontsize=9, ax=ax,
)
ax.set_title(
    f'Albero completo: max_depth={depth_to_plot}, profondità appresa={classifier_to_plot.get_depth()}\n'
    f'Vista fino a profondità {plot_max_depth}; (...) = sottoalberi nascosti, usati nelle predizioni'
)
fig.savefig(artifacts / 'decision_tree_demo.png', dpi=160)
plt.show()
print('Ordine delle proporzioni value:', classifier_to_plot.classes_.tolist())
np.testing.assert_array_equal(tree_to_plot.predict(X_validation), depth_predictions[depth_to_plot])


### Come leggere il disegno

Nei nodi interni, la prima riga è una condizione: vera → sinistra, falsa → destra.
`numeric__...` identifica una feature numerica; `categorical__...` un indicatore one-hot.
`gini` misura la mescolanza delle famiglie (0 = una sola famiglia); `samples` indica
la percentuale del train che raggiunge il nodo; `value` le proporzioni nell'ordine
`classes_`; `class` la famiglia prevalente. I valori sono arrotondati: `0.0` può
nascondere pochi esempi. I riquadri `(...)` sono rami nascosti, non foglie del modello.


## Interpretazione e limiti

### Esito misurato sullo stesso split

- **R2L:** passando da 10 a 13/15, gli errori scendono da **43/199 (21,61%)**
  a **14/199 (7,04%)**. Questi errori sono classificati come Normal.
- **U2R:** restano **3/10 errori (30%)**, tutti classificati come Normal, a ogni profondità.
  Aumentare la profondità fino a 15 non risolve questi casi su questa validation.
- **Falsi positivi U2R:** profondità 13 prevede U2R per 11 record, di cui 7 corretti;
  profondità 10 e 15 lo prevedono per 8 record, di cui 7 corretti. La precision U2R
  è quindi **63,64%** a 13, contro **87,50%** a 10/15. Anche una quota visualizzata
  come 0,0% nella matrice normalizzata può contenere errori: consultare i conteggi.
- **Macro F1:** 0,928445 a 10; 0,922114 a 13; **0,944019 a 15**. L'accuracy
  sale da 99,6547% a 99,758% a profondità 15, ma non risolve il limite U2R.

Fra le tre prove, 15 ha il macro F1 più alto; non è ancora una scelta finale del modello.
I risultati sono ripetibili su questo split, con due kernel puliti, non una stima del test.


Leggere prima le confusion matrix normalizzate e il riepilogo R2L/U2R, poi macro F1,
precision e metriche aggregate. La diagonale normalizzata misura il recall: una quota
fuori diagonale del 30% significa che il 30% degli esempi veri di quella famiglia
viene classificato altrove. La precision aiuta anche a capire se riconoscere più casi
introduce più falsi positivi.

U2R ha soltanto 10 esempi di validation: ogni errore vale 10 punti percentuali;
R2L ne ha 199. Migliorare su questo split non dimostra generalizzazione a nuovi dati.
Stiamo esplorando max_depth sulla stessa validation, senza una valutazione finale
sul test ufficiale. Non facciamo ulteriori cambiamenti prima della revisione dell'utente.

Tutte le 41 feature sono conservate. L'esperimento con 40 feature è stato rimosso.
Overlap e duplicati sono descritti nell'audit. Nessuna replica numerica di Divekar
e nessuna metrica del test ufficiale.
